# LeetCode #211: Design Add and Search Words Data Structure

https://leetcode.com/problems/design-add-and-search-words-data-structure/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (store words in list)** | $O(m \times n)$ per search | $O(n \times m)$ |
| **Optimal: Trie with DFS** | $O(m)$ add, $O(26^m)$ worst search | $O(\text{total chars})$ |

---

## Understanding the Methods

### Brute Force
Store all words in a list. For each search, iterate through all words and check if any matches the pattern (handling `.` as a wildcard). This is $O(m \times n)$ per search where `n` is the number of stored words and `m` is the word length.

### Optimal: Trie with DFS ★
Build a standard trie (prefix tree) for `addWord`. Each node has up to 26 children and an `isEnd` flag.

For `search` with wildcard `.`:
- Non-dot characters: follow the corresponding child.
- Dot character: try all 26 possible children recursively. If any path leads to a match, return true.

**Why this works:** The trie prunes the search space by sharing prefixes. Without dots, search is $O(m)$. With dots, we may explore multiple branches, but in practice the trie structure limits the search significantly compared to brute force.

**Constraints:**
* `1 <= word.length <= 25`
* Words contain lowercase letters; search patterns contain lowercase letters and `.`.
* At most 10^4 calls to `addWord` and `search`.

## Solutions

### C#

In [ ]:
public class WordDictionary {
    private class TrieNode {
        public TrieNode[] Children = new TrieNode[26];
        public bool IsEnd;
    }
    
    private TrieNode root;
    
    public WordDictionary() {
        root = new TrieNode();
    }
    
    public void AddWord(string word) {
        var node = root;
        foreach (char c in word) {
            int idx = c - 'a';
            if (node.Children[idx] == null)
                node.Children[idx] = new TrieNode();
            node = node.Children[idx];
        }
        node.IsEnd = true;
    }
    
    public bool Search(string word) {
        return Dfs(word, 0, root);
    }
    
    private bool Dfs(string word, int i, TrieNode node) {
        if (node == null) return false;
        if (i == word.Length) return node.IsEnd;
        
        if (word[i] == '.') {
            foreach (var child in node.Children) {
                if (Dfs(word, i + 1, child)) return true;
            }
            return false;
        }
        
        return Dfs(word, i + 1, node.Children[word[i] - 'a']);
    }
}

### Python

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False

class WordDictionary:
    def __init__(self):
        self.root = TrieNode()
    
    def addWord(self, word: str) -> None:
        node = self.root
        for c in word:
            if c not in node.children:
                node.children[c] = TrieNode()
            node = node.children[c]
        node.is_end = True
    
    def search(self, word: str) -> bool:
        def dfs(i, node):
            if i == len(word):
                return node.is_end
            if word[i] == '.':
                return any(dfs(i + 1, child) for child in node.children.values())
            if word[i] not in node.children:
                return False
            return dfs(i + 1, node.children[word[i]])
        
        return dfs(0, self.root)

### Go

In [ ]:
type TrieNode struct {
    Children [26]*TrieNode
    IsEnd    bool
}

type WordDictionary struct {
    Root *TrieNode
}

func Constructor() WordDictionary {
    return WordDictionary{Root: &TrieNode{}}
}

func (wd *WordDictionary) AddWord(word string) {
    node := wd.Root
    for _, c := range word {
        idx := c - 'a'
        if node.Children[idx] == nil {
            node.Children[idx] = &TrieNode{}
        }
        node = node.Children[idx]
    }
    node.IsEnd = true
}

func (wd *WordDictionary) Search(word string) bool {
    return dfs(word, 0, wd.Root)
}

func dfs(word string, i int, node *TrieNode) bool {
    if node == nil {
        return false
    }
    if i == len(word) {
        return node.IsEnd
    }
    if word[i] == '.' {
        for _, child := range node.Children {
            if dfs(word, i+1, child) {
                return true
            }
        }
        return false
    }
    return dfs(word, i+1, node.Children[word[i]-'a'])
}

### Rust

In [ ]:
struct TrieNode {
    children: [Option<Box<TrieNode>>; 26],
    is_end: bool,
}

impl TrieNode {
    fn new() -> Self {
        TrieNode {
            children: Default::default(),
            is_end: false,
        }
    }
}

struct WordDictionary {
    root: TrieNode,
}

impl WordDictionary {
    fn new() -> Self {
        WordDictionary { root: TrieNode::new() }
    }
    
    fn add_word(&mut self, word: String) {
        let mut node = &mut self.root;
        for b in word.bytes() {
            let idx = (b - b'a') as usize;
            node = node.children[idx].get_or_insert_with(|| Box::new(TrieNode::new()));
        }
        node.is_end = true;
    }
    
    fn search(&self, word: String) -> bool {
        Self::dfs(word.as_bytes(), 0, &self.root)
    }
    
    fn dfs(word: &[u8], i: usize, node: &TrieNode) -> bool {
        if i == word.len() {
            return node.is_end;
        }
        if word[i] == b'.' {
            for child in &node.children {
                if let Some(c) = child {
                    if Self::dfs(word, i + 1, c) {
                        return true;
                    }
                }
            }
            return false;
        }
        let idx = (word[i] - b'a') as usize;
        match &node.children[idx] {
            Some(child) => Self::dfs(word, i + 1, child),
            None => false,
        }
    }
}

## Example Scenarios

1. **Exact match:** Add `"bad"`, search `"bad"` → true. Direct trie traversal.

2. **Wildcard match:** Add `"bad"`, search `".ad"` → true. Dot matches `'b'`.

3. **No match:** Add `"bad"`, search `"bae"` → false. Last character differs.

4. **All wildcards:** Add `"bad"`, search `"..."` → true. Three dots match any 3-letter word.

5. **Prefix not a word:** Add `"bad"`, search `"ba"` → false. `"ba"` is a prefix but not a complete word in the trie.

*Infographic will be added in a future update.*